# Phase 0 - Week 2 - Day 3 PM - Relational Database and Queries: Google BigQuery

## A. Google BigQuery Hierarchy

Like other Google Cloud services, Google BigQuery resources are organized in a hierarchy. You can use this hierarchy to manage aspects of your BigQuery workloads such as permissions, quotas, slot reservations, and billing. Here is the breakdown of the BigQuery data hierarchy:

1. `Project` (The Root)
    - The `Project` is the highest level of the hierarchy. It acts as the "container" for everything else.
    - Purpose: It manages billing and permissions (IAM).
    - Key Fact: All queries you run and all data you store are billed to a specific Project ID.

2. `Dataset` (The Logical Folder)
    - A `Dataset` will contain tables, views, functions, and procedures.
    - A `Dataset` is a container used to organize and control access to your tables and views.
    - Purpose: It acts like a "Schema" in traditional SQL databases.
    - Location: This is where you define the Data Location (e.g., US, EU, or Tokyo). You cannot move a dataset to a different region once it is created.
    - Permissions: You usually set access rights (who can read/write) at this level.

3. `Table`/`View` (The Data Level)
    - This is where your actual data lives.
    - `Tables`: Collections of individual records organized into rows and columns.
    - `Views` : Virtual tables defined by a SQL query. They don't store data themselves but represent a "saved search."
    - External Tables: Links to data stored outside BigQuery (like a CSV in Google Cloud Storage or a Google Sheet).

4. `Job` (The Operational Level)
    - While not a storage unit, `Jobs` are actions BigQuery performs on your behalf.
    - Purpose: Every time you run a query, load data, or export data, BigQuery creates a `Job ID`.
    - History: You can look back at your Job History to see how much data a specific query processed or if it failed.

## B. `bigquery-public-data`

The public datasets are datasets that BigQuery hosts for you to access and available to the general public. Google pays for the storage of these datasets and provides public access to the data using a project. You pay only for the queries that you perform on the data. The first 1 TB per month is free, subject to query pricing details. **One of the public dataset is `bigquery-public-data` that contains several datasets and tables.**

Steps to access `bigquery-public-data` :

1. In order for `bigquery-public-data` to be visible in your Google BigQuery console, please visit one of its datasets, such as the [Austin Crime Dataset](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=austin_crime&page=dataset).

2. `bigquery-public-data` should now be visible in your window. Click the star icon next to it to make sure it appears every time you access [Google BigQuery](https://console.cloud.google.com/bigquery).

    <img src="https://i.ibb.co.com/TM12QgR6/Google-Big-Query-bigquery-public-data.png" height="auto">

3. In the displayed `austin_crime` dataset, you can see that it contains a table named `crime`. Click on this table to see its contents.

4. Here is the Google BigQuery hierarchy up to this point:
    ```
    Project ID: bigquery-public-data
    Dataset: austin_crime
    Table: crime
    ```

## C. Data Query Language (DQL) on Google BigQuery

To start querying, click the `+` button to open a new SQL editor window.

<img src="https://i.ibb.co.com/wh4r4kpM/Google-Big-Query-Add-SQL-Window.png">

Google BigQuery uses SQL syntax that's very similar to what you've learned with PostgreSQL. While there are a few differences, they share most of the same syntax.

*Check out [this link](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax) for more details on BigQuery syntax.*

One of the differences when querying in BigQuery is how you declare the `FROM` clause. In this section, you must write it in the format **\`Project-ID.Dataset.Table\`**. You must include backticks (\`) when declaring the `FROM` clause.

As a first step, let's display all the data in the crime table.
```
Project ID: bigquery-public-data
Dataset: austin_crime
Table: crime
```

The SQL syntax will be:
```sql
SELECT *
FROM `bigquery-public-data.austin_crime.crime`;
```

Here are some queries you can try:

1. Count total.
    
    Find out how many crime reports are in the entire dataset

    ```sql
    SELECT COUNT(*) AS total_crimes
    FROM `bigquery-public-data.austin_crime.crime`;
    ```

2. List unique crime types.
    
    Identify the different categories of crimes recorded in Austin.

    ```sql
    SELECT DISTINCT primary_type
    FROM `bigquery-public-data.austin_crime.crime`
    ORDER BY primary_type ASC;
    ```

3. Top 10 most common crimes.
    
    Which crimes happen most frequently?

    ```sql
    SELECT primary_type, COUNT(*) AS crime_count
    FROM `bigquery-public-data.austin_crime.crime`
    GROUP BY primary_type
    ORDER BY crime_count DESC
    LIMIT 10;
    ```

4. Crimes by year.
    
    Analyze the trend of crime over time.

    ```sql
    SELECT year, COUNT(*) AS yearly_crimes
    FROM `bigquery-public-data.austin_crime.crime`
    GROUP BY year
    ORDER BY year DESC;
    ```

5. Monthly crime trends in `2016`.
    
    Does crime increase in certain months? We use `EXTRACT` to pull the month from the timestamp.

    ```sql
    SELECT EXTRACT(MONTH FROM timestamp) AS month, COUNT(*) AS crime_count
    FROM `bigquery-public-data.austin_crime.crime`
    WHERE year=2016
    GROUP BY month
    ORDER BY month ASC;
    ```

## D. Query on Google Colab

You can run SQL queries on data from Google BigQuery using Google Colab. This means you can combine SQL syntax with Python syntax within Google Colab. **You'll need the ID of the Google Cloud Platform project you created earlier.**

> ***Remember you need GCP Project ID not GCP Project Name***

Here are the steps to find your GCP Project ID:

1. Click on the GCP Project Name you created earlier.

    You can find your GCP Project Name at the top of the Google Cloud page. For example, the image below uses the GCP Project Name `Personal-Danu`.

    <img src="https://i.ibb.co.com/ch4rMFY9/Google-Big-Query-GCP-Project-Name.png">

2. A pop-up window will appear showing your GCP Project Name and GCP Project ID. The GCP Project ID is usually in lowercase.

    <img src="https://i.ibb.co.com/23JfRvR8/Google-Big-Query-GCP-Project-ID.png">

    Summary from the image above :
    ```
    GCP Project Name: Personal-Danu
    GCP Project ID: personal-danu
    ```

In [1]:
# Connecting Google Colab with Google Cloud

from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()
print('Authenticated')

GCP_PROJECT_ID = 'personal-danu' # Use your GCP Project ID
client = bigquery.Client(project = GCP_PROJECT_ID)

Authenticated


When you run the syntax above, a pop-up window will appear asking for verification. Follow the verification steps as prompted in the window.

To run an SQL query, you can write it inside the following Python code format:

```py
df = client.query('''
<your-sql-query>
''').to_dataframe()
```

In [2]:
# Example : Find out how many crime reports are in the entire dataset

df = client.query('''
SELECT COUNT(*) AS total_crimes
FROM `bigquery-public-data.austin_crime.crime`;
''').to_dataframe()

df

,total_crimes
0,116672


In [3]:
# Another example : Identify the different categories of crimes recorded in Austin.

df = client.query('''
SELECT DISTINCT primary_type
FROM `bigquery-public-data.austin_crime.crime`
ORDER BY primary_type ASC;
''').to_dataframe()

df

,primary_type
0,Agg Assault
1,Aggravated Assault
2,Auto Theft
3,Burglary
4,Burglary / \nBreaking & Entering
5,Homicide: Murder & Nonnegligent Manslaughter
6,Murder
7,Rape
8,Robbery
9,Theft
